In [ ]:
# Step 1: Ingestion & Training Execution
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training Pipeline Device: {device}")

class FullMultimodalFusionSystem(nn.Module):
    def __init__(self, embed_dim=256, num_classes=3):
        super().__init__()
        self.proj_text = nn.Linear(768, embed_dim)
        self.proj_audio = nn.Linear(16, embed_dim)
        self.proj_image = nn.Linear(768, embed_dim)
        self.attention = nn.MultiheadAttention(embed_dim, num_heads=4, batch_first=True)
        self.fc1 = nn.Linear(embed_dim, 128)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, t, a, v):
        proj_t = self.proj_text(t)
        proj_a = self.proj_audio(a)
        proj_v = self.proj_image(v)
        stacked = torch.stack([proj_t, proj_a, proj_v], dim=1)
        attn_out, _ = self.attention(stacked, stacked, stacked)
        fused = torch.mean(attn_out, dim=1)
        x = self.dropout(self.relu(self.fc1(fused)))
        return self.classifier(x)

# Dataset Generation
np.random.seed(42)
num_samples = 2400
labels = np.random.choice([0, 1, 2], size=num_samples, p=[0.35, 0.30, 0.35])
t_data = torch.randn(num_samples, 768)
a_data = torch.randn(num_samples, 16)
v_data = torch.randn(num_samples, 768)
y_data = torch.tensor(labels, dtype=torch.long)

dataset = torch.utils.data.TensorDataset(t_data, a_data, v_data, y_data)
train_ds, val_ds = torch.utils.data.random_split(dataset, [2000, 400])

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)

model = FullMultimodalFusionSystem().to(device)
optimizer = AdamW(model.parameters(), lr=2e-4, weight_decay=1e-2)
criterion = nn.CrossEntropyLoss()

epochs = 8
train_losses, val_losses = [], []
train_accs, val_accs = [], []

for epoch in range(1, epochs + 1):
    model.train()
    t_loss, correct, total = 0.0, 0, 0
    for t, a, v, y in train_loader:
        t, a, v, y = t.to(device), a.to(device), v.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(t, a, v)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        
        t_loss += loss.item() * y.size(0)
        correct += (torch.argmax(logits, dim=1) == y).sum().item()
        total += y.size(0)
        
    tr_loss = t_loss / total
    tr_acc = (correct / total) * 100
    train_losses.append(tr_loss)
    train_accs.append(tr_acc)
    
    model.eval()
    v_loss, v_correct, v_total = 0.0, 0, 0
    with torch.no_grad():
        for t, a, v, y in val_loader:
            t, a, v, y = t.to(device), a.to(device), v.to(device), y.to(device)
            logits = model(t, a, v)
            loss = criterion(logits, y)
            v_loss += loss.item() * y.size(0)
            v_correct += (torch.argmax(logits, dim=1) == y).sum().item()
            v_total += y.size(0)
            
    va_loss = v_loss / v_total
    va_acc = (v_correct / v_total) * 100
    val_losses.append(va_loss)
    val_accs.append(va_acc)
    
    print(f"Epoch [{epoch}/{epochs}] - Train Loss: {tr_loss:.4f} | Train Acc: {tr_acc:.2f}% | Val Loss: {va_loss:.4f} | Val Acc: {va_acc:.2f}%")

In [ ]:
# Step 2: Full Pipeline Loss & Accuracy Curve Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(range(1, epochs + 1), train_losses, label='Train Loss', marker='o', color='#dc2626', linewidth=2)
axes[0].plot(range(1, epochs + 1), val_losses, label='Validation Loss', marker='s', color='#d97706', linewidth=2)
axes[0].set_title('Pipeline Training & Validation Loss', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epochs')
axes[0].set_ylabel('Loss')
axes[0].grid(True, linestyle='--', alpha=0.6)
axes[0].legend()

axes[1].plot(range(1, epochs + 1), train_accs, label='Train Accuracy', marker='o', color='#2563eb', linewidth=2)
axes[1].plot(range(1, epochs + 1), val_accs, label='Validation Accuracy', marker='s', color='#059669', linewidth=2)
axes[1].set_title('Pipeline Training & Validation Accuracy', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epochs')
axes[1].set_ylabel('Accuracy (%)')
axes[1].grid(True, linestyle='--', alpha=0.6)
axes[1].legend()

plt.tight_layout()
plt.show()